# Notebook 02 — Monte Carlo Confidence Intervals

Run 500 replications and compute confidence intervals for MCR and FMCR.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

from src.simulation import FleetSimulation, run_monte_carlo

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.dpi'] = 120

print('Ready.')

## Run 500 Replications

In [ ]:
config = {
    'fleet_size': 12,
    'o_level_techs_day': 8,
    'i_level_techs_day': 4,
    'd_level_techs': 6,
    'sortie_interval_hours': 48,
}

mc_df = run_monte_carlo(
    config, n_replications=500,
    simulation_days=180, random_seed=42
)

print(f'Completed {len(mc_df)} replications.')
mc_df.describe()

## 1. Histogram of MCR with Normal Overlay

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
n, bins, patches = ax.hist(mc_df['mcr'] * 100, bins=30, density=True,
                           alpha=0.7, color='#3498db', edgecolor='white')

# Normal overlay
mu = mc_df['mcr'].mean() * 100
sigma = mc_df['mcr'].std() * 100
x = np.linspace(mu - 4*sigma, mu + 4*sigma, 100)
ax.plot(x, norm.pdf(x, mu, sigma), 'r-', linewidth=2,
        label=f'Normal fit (μ={mu:.1f}%, σ={sigma:.2f}%)')

ax.set_xlabel('MCR (%)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Distribution of MCR Across 500 Replications', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Histogram of FMCR

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(mc_df['fmcr'] * 100, bins=30, density=True,
        alpha=0.7, color='#2ecc71', edgecolor='white')

mu_f = mc_df['fmcr'].mean() * 100
sigma_f = mc_df['fmcr'].std() * 100
x_f = np.linspace(mu_f - 4*sigma_f, mu_f + 4*sigma_f, 100)
ax.plot(x_f, norm.pdf(x_f, mu_f, sigma_f), 'r-', linewidth=2,
        label=f'Normal fit (μ={mu_f:.1f}%, σ={sigma_f:.2f}%)')

ax.set_xlabel('FMCR (%)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Distribution of FMCR Across 500 Replications', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 3. Convergence Plot

In [ ]:
running_mean = mc_df['mcr'].expanding().mean() * 100
running_std = mc_df['mcr'].expanding().std()
n_vals = np.arange(1, len(mc_df) + 1)
ci_half = 1.96 * running_std * 100 / np.sqrt(n_vals)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(n_vals, running_mean, color='#2980b9', linewidth=2, label='Running Mean MCR')
ax.fill_between(n_vals, running_mean - ci_half, running_mean + ci_half,
                alpha=0.2, color='#2ecc71', label='95% CI')
ax.set_xlabel('Number of Replications', fontsize=12)
ax.set_ylabel('MCR (%)', fontsize=12)
ax.set_title('Convergence of Monte Carlo Estimate', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Confidence Intervals

In [ ]:
n = len(mc_df)
mcr_mean = mc_df['mcr'].mean()
mcr_se = mc_df['mcr'].std() / np.sqrt(n)
fmcr_mean = mc_df['fmcr'].mean()
fmcr_se = mc_df['fmcr'].std() / np.sqrt(n)

print('95% Confidence Intervals (500 replications)')
print('=' * 55)
print(f'MCR:  {mcr_mean*100:.2f}% ± {1.96*mcr_se*100:.2f}%')
print(f'      [{(mcr_mean - 1.96*mcr_se)*100:.2f}%, {(mcr_mean + 1.96*mcr_se)*100:.2f}%]')
print(f'FMCR: {fmcr_mean*100:.2f}% ± {1.96*fmcr_se*100:.2f}%')
print(f'      [{(fmcr_mean - 1.96*fmcr_se)*100:.2f}%, {(fmcr_mean + 1.96*fmcr_se)*100:.2f}%]')

## 5. Minimum Replications for CI Half-Width of 0.01

In [ ]:
# Required n for CI half-width of 1 percentage point (0.01)
target_halfwidth = 0.01
s = mc_df['mcr'].std()
n_required = int(np.ceil((1.96 * s / target_halfwidth) ** 2))

print(f'Sample std of MCR: {s:.4f}')
print(f'Minimum replications for CI half-width of {target_halfwidth*100:.0f}%: {n_required}')